# Task 2: MySQL and MongoDB Database Design
## Beijing Multi-Site Air Quality Dataset

**Member 2** | Builds on Task 1 (preprocessing, EDA, and ML model)

---

### Overview

This notebook designs and implements two databases for the Beijing Air Quality time-series dataset:

| Component | Technology | Contents |
|-----------|-----------|----------|
| Relational DB | MySQL schema · executed via SQLite | 3 normalized tables, 420 768 hourly rows |
| Document DB | MongoDB schema · executed via mongomock | 1 collection, 420 768 documents |

All schema scripts are written in **MySQL syntax**. SQLite is used purely as the execution engine so the notebook runs without a server; the SQL syntax is identical for every construct used here.

> **Upgrading to a real MongoDB on Kaggle**: uncomment the two bash lines in the MongoDB setup cell to install and start a local MongoDB process — the pymongo code that follows is unchanged.

---
## Section 1: Load Dataset

In [ ]:
# Install mongomock (pymongo-compatible in-process MongoDB)
!pip install mongomock --quiet

import os, sqlite3, glob, json, math, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from datetime import datetime
import mongomock

warnings.filterwarnings('ignore')
print('All packages ready.')

In [ ]:
# ── Locate the 12 station CSV files ──────────────────────────────────────────
# In Kaggle the dataset is pre-extracted under /kaggle/input/
csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)

if not csv_files:
    raise FileNotFoundError(
        'No CSV files found under /kaggle/input/.\n'
        'Add the Beijing Multi-Site Air Quality dataset to this notebook.'
    )

print(f'Found {len(csv_files)} CSV file(s):')
for f in sorted(csv_files):
    print(f'  {f}')

# Combine all station files into one DataFrame
df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Build a proper timestamp from the four date components
df_raw['timestamp'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])

print(f'\nDataset loaded: {len(df_raw):,} rows × {len(df_raw.columns)} columns')
print(f'Time range : {df_raw["timestamp"].min()}  →  {df_raw["timestamp"].max()}')
print(f'Stations   : {sorted(df_raw["station"].unique().tolist())}')

In [ ]:
print('Columns:', df_raw.columns.tolist())
print()
display(df_raw.head(3))
print(f'\nShape: {df_raw.shape}')
print('\nMissing values per column:')
display(df_raw.isnull().sum().rename('missing').to_frame())

---
## Section 2: Relational Database Schema Design

### Table Descriptions

| Table | Role | Row count |
|-------|------|-----------|
| `stations` | Master record for each monitoring station | 12 |
| `readings` | Hourly measurement fact table — all pollutant and weather values | 420 768 |
| `pollutant_reference` | WHO guideline lookup per pollutant | 6 |

### Design Rationale
- **stations** normalises the station name out of the 420 768-row fact table (12 rows instead of 420 768 repeated strings).
- **readings** holds the time-series core. A composite `UNIQUE (station_id, timestamp)` constraint guarantees no duplicate hourly records per station. Two indexes (`timestamp`, `station_id + timestamp`) support the date-range and latest-record queries that Task 3 will expose as API endpoints.
- **pollutant_reference** stores WHO 2021 air-quality guidelines per pollutant, enabling threshold queries without hardcoding values in application code. It is a lookup / reference table — no FK is added to `readings` because all six pollutants are always measured together in the same row.

In [ ]:
# ── Entity-Relationship Diagram ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(17, 10))
ax.set_xlim(0, 17)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('#f4f6f9')
fig.patch.set_facecolor('#f4f6f9')
ax.set_title('Entity-Relationship Diagram — Beijing Air Quality Database',
             fontsize=14, fontweight='bold', pad=14, color='#2c3e50')

COLOR = {'pk': '#c0392b', 'fk': '#d35400', 'none': '#7f8c8d',
          'stations': '#27ae60', 'readings': '#2980b9', 'ref': '#8e44ad'}

def draw_table(ax, x, y, w, h, title, rows, hdr_color):
    """Draw one entity box with a coloured header and attribute rows."""
    row_h = (h - 0.62) / max(len(rows), 1)

    # header
    hdr = FancyBboxPatch((x, y + h - 0.62), w, 0.62,
                         boxstyle='round,pad=0.04',
                         facecolor=hdr_color, edgecolor='#1a252f', linewidth=1.8)
    ax.add_patch(hdr)
    ax.text(x + w / 2, y + h - 0.31, title,
            ha='center', va='center', fontsize=11, fontweight='bold', color='white')

    # body
    body = FancyBboxPatch((x, y), w, h - 0.62,
                          boxstyle='round,pad=0.04',
                          facecolor='white', edgecolor='#1a252f', linewidth=1.8)
    ax.add_patch(body)

    for i, (ktype, col, dtype) in enumerate(rows):
        ry = y + h - 0.62 - (i + 0.5) * row_h

        if ktype == 'PK':
            badge_color, symbol = COLOR['pk'], 'PK'
        elif ktype == 'FK':
            badge_color, symbol = COLOR['fk'], 'FK'
        else:
            badge_color, symbol = COLOR['none'], ''

        if symbol:
            badge = FancyBboxPatch((x + 0.07, ry - 0.14), 0.37, 0.28,
                                   boxstyle='round,pad=0.02',
                                   facecolor=badge_color, edgecolor='none')
            ax.add_patch(badge)
            ax.text(x + 0.255, ry, symbol,
                    ha='center', va='center', fontsize=6.5, fontweight='bold', color='white')

        ax.text(x + 0.52, ry, col,
                ha='left', va='center', fontsize=8.5,
                color='#2c3e50', fontfamily='monospace')
        ax.text(x + w - 0.1, ry, dtype,
                ha='right', va='center', fontsize=7.5, color='#7f8c8d', style='italic')

        if i < len(rows) - 1:
            sep_y = ry - row_h / 2
            ax.plot([x + 0.07, x + w - 0.07], [sep_y, sep_y],
                    color='#ecf0f1', linewidth=0.7)

# ── stations (left) ──────────────────────────────────────────────────────────
stations_rows = [
    ('PK', 'station_id',   'INT'),
    ('',   'station_name', 'VARCHAR(50)'),
    ('',   'city',         'VARCHAR(100)'),
    ('',   'created_at',   'TIMESTAMP'),
]
draw_table(ax, 0.4, 5.2, 4.4, 3.5, 'stations', stations_rows, COLOR['stations'])

# ── readings (centre) ────────────────────────────────────────────────────────
readings_rows = [
    ('PK', 'reading_id',     'BIGINT'),
    ('FK', 'station_id',     'INT'),
    ('',   'timestamp',      'DATETIME'),
    ('',   'pm25',           'FLOAT'),
    ('',   'pm10',           'FLOAT'),
    ('',   'so2',            'FLOAT'),
    ('',   'no2',            'FLOAT'),
    ('',   'co',             'FLOAT'),
    ('',   'o3',             'FLOAT'),
    ('',   'temp',           'FLOAT'),
    ('',   'pres',           'FLOAT'),
    ('',   'dewp',           'FLOAT'),
    ('',   'rain',           'FLOAT'),
    ('',   'wind_direction', 'VARCHAR(10)'),
    ('',   'wind_speed',     'FLOAT'),
]
draw_table(ax, 5.8, 0.3, 5.4, 9.4, 'readings', readings_rows, COLOR['readings'])

# ── pollutant_reference (right) ──────────────────────────────────────────────
ref_rows = [
    ('PK', 'pollutant_id',     'INT'),
    ('',   'code',             'VARCHAR(20)'),
    ('',   'full_name',        'VARCHAR(100)'),
    ('',   'unit',             'VARCHAR(20)'),
    ('',   'who_annual_limit', 'FLOAT'),
    ('',   'who_24h_limit',    'FLOAT'),
    ('',   'description',      'TEXT'),
]
draw_table(ax, 11.9, 4.0, 4.7, 5.6, 'pollutant_reference', ref_rows, COLOR['ref'])

# ── Relationship line: stations (1) ——< readings (N) ─────────────────────────
# horizontal line from right edge of stations to left edge of readings
ax.annotate('', xy=(5.8, 6.95), xytext=(4.8, 6.95),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.0))
ax.text(4.85, 7.22, '1', fontsize=12, fontweight='bold', color='#2c3e50')
ax.text(5.55, 7.22, 'N', fontsize=12, fontweight='bold', color='#2c3e50')
ax.text(4.9, 6.62, 'has', fontsize=8, color='#555', style='italic')

# ── Conceptual reference: pollutant_reference ↔ readings column names ────────
ax.annotate('', xy=(11.2, 5.8), xytext=(11.9, 5.8),
            arrowprops=dict(arrowstyle='<->', color=COLOR['ref'], lw=1.5,
                            linestyle='dashed'))
ax.text(10.95, 6.05, 'column', fontsize=7, color=COLOR['ref'], style='italic')
ax.text(10.85, 5.75, 'reference', fontsize=7, color=COLOR['ref'], style='italic')

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=COLOR['pk'],       label='PK  Primary Key'),
    mpatches.Patch(facecolor=COLOR['fk'],       label='FK  Foreign Key'),
    mpatches.Patch(facecolor=COLOR['stations'], label='stations table'),
    mpatches.Patch(facecolor=COLOR['readings'], label='readings table'),
    mpatches.Patch(facecolor=COLOR['ref'],      label='pollutant_reference table'),
]
ax.legend(handles=legend_handles, loc='lower left', fontsize=8.5,
          framealpha=0.95, edgecolor='#bdc3c7')

plt.tight_layout()
plt.savefig('erd_beijing_air_quality.png', dpi=200, bbox_inches='tight',
            facecolor='#f4f6f9')
plt.show()
print('ERD saved → erd_beijing_air_quality.png')

---
## MySQL CREATE TABLE Scripts

The three statements below are the authoritative MySQL DDL for this schema.
They are printed here for the report and then re-executed (with minor SQLite-specific
adjustments) in the next section.

In [ ]:
mysql_ddl = """
-- =============================================================
--  Beijing Air Quality — MySQL Schema Scripts
--  Database: beijing_air_quality
-- =============================================================

-- Table 1: stations
--   Master record for each of the 12 monitoring stations.
CREATE TABLE IF NOT EXISTS stations (
    station_id    INT           NOT NULL AUTO_INCREMENT,
    station_name  VARCHAR(50)   NOT NULL,
    city          VARCHAR(100)  NOT NULL DEFAULT 'Beijing',
    created_at    TIMESTAMP     NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (station_id),
    UNIQUE KEY uq_station_name (station_name)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;


-- Table 2: readings
--   Hourly air-quality measurements from all 12 stations.
--   Each row is one station × one hour.
CREATE TABLE IF NOT EXISTS readings (
    reading_id      BIGINT        NOT NULL AUTO_INCREMENT,
    station_id      INT           NOT NULL,
    timestamp       DATETIME      NOT NULL,
    pm25            FLOAT                  COMMENT 'PM2.5  μg/m³',
    pm10            FLOAT                  COMMENT 'PM10   μg/m³',
    so2             FLOAT                  COMMENT 'SO2    μg/m³',
    no2             FLOAT                  COMMENT 'NO2    μg/m³',
    co              FLOAT                  COMMENT 'CO     μg/m³',
    o3              FLOAT                  COMMENT 'O3     μg/m³',
    temp            FLOAT                  COMMENT 'Temperature   °C',
    pres            FLOAT                  COMMENT 'Air pressure   hPa',
    dewp            FLOAT                  COMMENT 'Dew-point temperature  °C',
    rain            FLOAT                  COMMENT 'Precipitation  mm/h',
    wind_direction  VARCHAR(10)            COMMENT 'Compass direction (e.g. NNW)',
    wind_speed      FLOAT                  COMMENT 'Wind speed  m/s',
    PRIMARY KEY (reading_id),
    UNIQUE KEY uq_station_ts (station_id, timestamp),
    CONSTRAINT fk_readings_station
        FOREIGN KEY (station_id) REFERENCES stations (station_id)
        ON UPDATE CASCADE ON DELETE RESTRICT,
    INDEX idx_timestamp   (timestamp),
    INDEX idx_station_ts  (station_id, timestamp)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;


-- Table 3: pollutant_reference
--   WHO 2021 air-quality guideline values per measured pollutant.
--   Used in threshold queries (Query 5) without hardcoding limits.
CREATE TABLE IF NOT EXISTS pollutant_reference (
    pollutant_id       INT           NOT NULL AUTO_INCREMENT,
    code               VARCHAR(20)   NOT NULL,
    full_name          VARCHAR(100)  NOT NULL,
    unit               VARCHAR(20)   NOT NULL,
    who_annual_limit   FLOAT                  COMMENT 'WHO annual mean guideline (μg/m³)',
    who_24h_limit      FLOAT                  COMMENT 'WHO 24-hour mean guideline (μg/m³)',
    description        TEXT,
    PRIMARY KEY (pollutant_id),
    UNIQUE KEY uq_code (code)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

print(mysql_ddl)

---
## Section 3: SQL Implementation (SQLite)

> **SQLite vs MySQL**: SQLite uses `AUTOINCREMENT` instead of `AUTO_INCREMENT`, `TEXT`/`REAL` instead of `VARCHAR`/`FLOAT`, and `strftime()` instead of `DATE_FORMAT()` for date grouping. All other SQL in this notebook is MySQL-compatible.

In [ ]:
DB_PATH = '/kaggle/working/beijing_air_quality.db'

# Remove any previous run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
PRAGMA journal_mode = WAL;
PRAGMA synchronous  = NORMAL;

CREATE TABLE IF NOT EXISTS stations (
    station_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    station_name  TEXT    NOT NULL UNIQUE,
    city          TEXT    NOT NULL DEFAULT 'Beijing',
    created_at    TEXT    NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS readings (
    reading_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    station_id      INTEGER NOT NULL,
    timestamp       TEXT    NOT NULL,
    pm25            REAL,
    pm10            REAL,
    so2             REAL,
    no2             REAL,
    co              REAL,
    o3              REAL,
    temp            REAL,
    pres            REAL,
    dewp            REAL,
    rain            REAL,
    wind_direction  TEXT,
    wind_speed      REAL,
    FOREIGN KEY (station_id) REFERENCES stations (station_id),
    UNIQUE (station_id, timestamp)
);

CREATE INDEX IF NOT EXISTS idx_timestamp  ON readings (timestamp);
CREATE INDEX IF NOT EXISTS idx_station_ts ON readings (station_id, timestamp);

CREATE TABLE IF NOT EXISTS pollutant_reference (
    pollutant_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    code              TEXT  NOT NULL UNIQUE,
    full_name         TEXT  NOT NULL,
    unit              TEXT  NOT NULL,
    who_annual_limit  REAL,
    who_24h_limit     REAL,
    description       TEXT
);
""")

conn.commit()
print('Database created and tables initialised.')

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
print('Tables:', [r[0] for r in cursor.fetchall()])

In [ ]:
# ── 1. Insert all 12 stations ─────────────────────────────────────────────────
stations_list = sorted(df_raw['station'].unique())
cursor.executemany(
    'INSERT OR IGNORE INTO stations (station_name) VALUES (?)',
    [(s,) for s in stations_list]
)
conn.commit()
print(f'Inserted {len(stations_list)} stations.')

# ── 2. Insert WHO pollutant reference data (2021 guidelines) ─────────────────
# Source: WHO Global Air Quality Guidelines, 2021
pollutant_rows = [
    ('PM2.5', 'Fine Particulate Matter',   'μg/m³',  5.0,    15.0,
     'Particles ≤ 2.5 μm. Penetrate deep into lungs; linked to cardiovascular disease.'),
    ('PM10',  'Coarse Particulate Matter', 'μg/m³', 15.0,    45.0,
     'Particles ≤ 10 μm. Irritate the upper respiratory tract.'),
    ('SO2',   'Sulfur Dioxide',            'μg/m³', None,    40.0,
     'Emitted by burning fossil fuels; precursor to acid rain and PM2.5 formation.'),
    ('NO2',   'Nitrogen Dioxide',          'μg/m³', 10.0,    25.0,
     'Produced by traffic and combustion; irritates airways and contributes to smog.'),
    ('CO',    'Carbon Monoxide',           'μg/m³', None,  4000.0,
     'Colorless, odorless gas from incomplete combustion; reduces blood O₂ capacity.'),
    ('O3',    'Ground-level Ozone',        'μg/m³', None,   100.0,
     'Formed by photochemical reactions involving NO2 and VOCs; damages lung tissue.'),
]

cursor.executemany(
    '''
    INSERT OR IGNORE INTO pollutant_reference
        (code, full_name, unit, who_annual_limit, who_24h_limit, description)
    VALUES (?, ?, ?, ?, ?, ?)
    ''',
    pollutant_rows
)
conn.commit()
print(f'Inserted {len(pollutant_rows)} pollutant reference records.')

# Verify
cursor.execute('SELECT pollutant_id, code, full_name, who_annual_limit, who_24h_limit FROM pollutant_reference')
ref_df = pd.DataFrame(cursor.fetchall(), columns=['id','code','full_name','who_annual','who_24h'])
print()
display(ref_df)

In [ ]:
# ── Bulk-insert 420 768 readings ──────────────────────────────────────────────
# Build station name → id lookup
cursor.execute('SELECT station_id, station_name FROM stations')
station_map = {name: sid for sid, name in cursor.fetchall()}

# Prepare DataFrame
float_cols = ['PM2.5','PM10','SO2','NO2','CO','O3','TEMP','PRES','DEWP','RAIN','WSPM']
ins = df_raw[['station','timestamp','PM2.5','PM10','SO2','NO2','CO','O3',
              'TEMP','PRES','DEWP','RAIN','wd','WSPM']].copy()

ins['station_id']     = ins['station'].map(station_map)
ins['timestamp_str']  = ins['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Replace NaN with None (SQLite NULL)
for col in float_cols:
    ins[col] = ins[col].where(ins[col].notna(), other=None)
ins['wd'] = ins['wd'].where(ins['wd'].notna(), other=None)

records = ins[['station_id','timestamp_str','PM2.5','PM10','SO2','NO2','CO','O3',
               'TEMP','PRES','DEWP','RAIN','wd','WSPM']].values.tolist()

print(f'Inserting {len(records):,} rows …')
t0 = time.time()

cursor.executemany(
    '''
    INSERT OR IGNORE INTO readings
        (station_id, timestamp, pm25, pm10, so2, no2, co, o3,
         temp, pres, dewp, rain, wind_direction, wind_speed)
    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    ''',
    records
)
conn.commit()

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s  ({len(records)/elapsed:,.0f} rows/sec)')
cursor.execute('SELECT COUNT(*) FROM readings')
print(f'Total rows in readings table: {cursor.fetchone()[0]:,}')

In [ ]:
# Verify row counts per station
cursor.execute("""
    SELECT s.station_name, COUNT(r.reading_id) AS row_count
    FROM   readings r
    JOIN   stations s ON r.station_id = s.station_id
    GROUP  BY s.station_name
    ORDER  BY s.station_name
""")
verify_df = pd.DataFrame(cursor.fetchall(), columns=['station', 'row_count'])
display(verify_df)
print(f'\nTotal: {verify_df["row_count"].sum():,} rows across {len(verify_df)} stations')

---
## SQL Queries

Five queries are executed against the populated database. All are written in standard SQL compatible with MySQL.

| # | Description |
|---|-------------|
| Q1 | Latest recorded reading for each station |
| Q2 | All hourly readings within a specified date range |
| Q3 | Average, maximum, and missing-value count for PM2.5 per station |
| Q4 | Monthly average PM2.5 trend across all stations |
| Q5 | WHO 2021 PM2.5 guideline exceedance analysis per station |

In [ ]:
# ── Query 1: Latest recorded reading for each monitoring station ──────────────
q1 = """
SELECT
    s.station_name,
    r.timestamp                AS latest_timestamp,
    ROUND(r.pm25,  2)          AS pm25,
    ROUND(r.pm10,  2)          AS pm10,
    ROUND(r.temp,  1)          AS temp_celsius,
    r.wind_direction,
    ROUND(r.wind_speed, 1)     AS wind_speed_ms
FROM readings r
JOIN stations s ON r.station_id = s.station_id
JOIN (
    SELECT station_id, MAX(timestamp) AS max_ts
    FROM   readings
    GROUP  BY station_id
) latest
  ON  r.station_id = latest.station_id
  AND r.timestamp  = latest.max_ts
ORDER BY s.station_name;
"""

print('Query 1: Latest reading per station')
print('-' * 60)
print(q1)

result_q1 = pd.read_sql_query(q1, conn)
display(result_q1)
print(f'Rows returned: {len(result_q1)}')

In [ ]:
# ── Query 2: All hourly readings for Dongsi — 1–7 January 2016 ───────────────
# MySQL equivalent: replace strftime with DATE() or BETWEEN with datetime literals
q2 = """
SELECT
    s.station_name,
    r.timestamp,
    ROUND(r.pm25, 2)       AS pm25,
    ROUND(r.pm10, 2)       AS pm10,
    ROUND(r.so2,  2)       AS so2,
    ROUND(r.no2,  2)       AS no2,
    ROUND(r.temp, 1)       AS temp_celsius,
    r.wind_direction,
    ROUND(r.wind_speed, 1) AS wind_speed_ms
FROM readings r
JOIN stations s ON r.station_id = s.station_id
WHERE s.station_name = 'Dongsi'
  AND r.timestamp BETWEEN '2016-01-01 00:00:00'
                      AND '2016-01-07 23:00:00'
ORDER BY r.timestamp;
"""

print('Query 2: Date-range — Dongsi station, 1–7 January 2016')
print('-' * 60)
print(q2)

result_q2 = pd.read_sql_query(q2, conn)
print(f'Total rows returned: {len(result_q2)}')
print('\nFirst 10 rows:')
display(result_q2.head(10))

In [ ]:
# ── Query 3: Average, maximum, and missing PM2.5 per station ─────────────────
q3 = """
SELECT
    s.station_name,
    ROUND(AVG(r.pm25), 2)                                   AS avg_pm25,
    ROUND(MAX(r.pm25), 2)                                   AS max_pm25,
    COUNT(r.reading_id)                                     AS total_readings,
    SUM(CASE WHEN r.pm25 IS NULL THEN 1 ELSE 0 END)        AS missing_pm25,
    ROUND(
        100.0 * SUM(CASE WHEN r.pm25 IS NULL THEN 1 ELSE 0 END)
              / COUNT(r.reading_id), 1
    )                                                       AS pct_missing
FROM readings r
JOIN stations s ON r.station_id = s.station_id
GROUP BY s.station_name
ORDER BY avg_pm25 DESC;
"""

print('Query 3: Average PM2.5 by station (ranked highest → lowest)')
print('-' * 60)
print(q3)

result_q3 = pd.read_sql_query(q3, conn)
display(result_q3)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(result_q3['station_name'][::-1], result_q3['avg_pm25'][::-1],
               color='#3498db', edgecolor='#2980b9')
ax.axvline(x=15, color='#c0392b', linestyle='--', linewidth=1.5,
           label='WHO 24 h guideline (15 μg/m³)')
ax.axvline(x=5,  color='#e67e22', linestyle=':',  linewidth=1.5,
           label='WHO annual guideline (5 μg/m³)')
ax.set_xlabel('Average PM2.5 Concentration (μg/m³)')
ax.set_title('Average PM2.5 by Monitoring Station — 2013–2017')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('sql_q3_avg_pm25_by_station.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Query 4: Monthly average PM2.5 trend ─────────────────────────────────────
# MySQL equivalent: DATE_FORMAT(r.timestamp, '%Y-%m')
q4 = """
SELECT
    strftime('%Y-%m', r.timestamp)  AS year_month,
    ROUND(AVG(r.pm25), 2)           AS avg_pm25,
    ROUND(AVG(r.temp), 2)           AS avg_temp_celsius,
    COUNT(r.reading_id)             AS hourly_obs
FROM readings r
WHERE r.pm25 IS NOT NULL
GROUP BY strftime('%Y-%m', r.timestamp)
ORDER BY year_month;
"""

print('Query 4: Monthly average PM2.5 trend (all stations)')
print('-' * 60)
print(q4)

result_q4 = pd.read_sql_query(q4, conn)
display(result_q4)

# Dual-axis chart
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(result_q4['year_month'], result_q4['avg_pm25'],
         color='#c0392b', marker='o', markersize=3, linewidth=1.5, label='Avg PM2.5')
ax1.axhline(y=15, color='darkred', linestyle='--', alpha=0.6, label='WHO 24 h limit')
ax1.set_ylabel('Avg PM2.5 (μg/m³)', color='#c0392b')
ax1.tick_params(axis='x', rotation=55, labelsize=7)
ax1.legend(loc='upper left', fontsize=9)

ax2 = ax1.twinx()
ax2.fill_between(result_q4['year_month'], result_q4['avg_temp_celsius'],
                 alpha=0.18, color='#3498db')
ax2.plot(result_q4['year_month'], result_q4['avg_temp_celsius'],
         color='#2980b9', linewidth=1.2, label='Avg Temp (°C)')
ax2.set_ylabel('Avg Temp (°C)', color='#2980b9')
ax2.legend(loc='upper right', fontsize=9)

plt.title('Monthly Average PM2.5 and Temperature — All Stations (2013–2017)')
plt.tight_layout()
plt.savefig('sql_q4_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Query 5: WHO PM2.5 guideline exceedance analysis ─────────────────────────
# Joins readings against pollutant_reference so threshold values are not hardcoded.
q5 = """
SELECT
    s.station_name,
    pr.who_24h_limit,
    pr.who_annual_limit,
    COUNT(r.reading_id)                                               AS valid_readings,
    SUM(CASE WHEN r.pm25 > pr.who_24h_limit  THEN 1 ELSE 0 END)     AS hours_above_24h,
    ROUND(
        100.0 * SUM(CASE WHEN r.pm25 > pr.who_24h_limit THEN 1 ELSE 0 END)
              / COUNT(r.reading_id), 1
    )                                                                 AS pct_above_24h,
    ROUND(AVG(r.pm25), 2)                                            AS mean_pm25,
    CASE
        WHEN AVG(r.pm25) > pr.who_annual_limit THEN 'EXCEEDS annual'
        ELSE 'Within annual'
    END                                                               AS annual_status
FROM readings r
JOIN stations s ON r.station_id = s.station_id
CROSS JOIN pollutant_reference pr
WHERE r.pm25 IS NOT NULL
  AND pr.code  = 'PM2.5'
GROUP BY s.station_name, pr.who_24h_limit, pr.who_annual_limit
ORDER BY pct_above_24h DESC;
"""

print('Query 5: WHO PM2.5 guideline exceedance by station')
print('-' * 60)
print(q5)

result_q5 = pd.read_sql_query(q5, conn)
display(result_q5)

---
## Section 4: MongoDB Database Design

### Collection: `air_quality_readings`

Each document represents **one hourly observation at one station**.
Pollutant measurements and weather readings are stored in **nested sub-documents**
to reflect their natural grouping and to allow selective field projection in queries.

#### Document Schema

```json
{
  "_id": ObjectId("..."),
  "station":   "Dongsi",
  "timestamp": ISODate("2013-03-01T00:00:00Z"),
  "year": 2013, "month": 3, "day": 1, "hour": 0,
  "pollutants": {
    "pm25": 9.0,  "pm10": 9.0,  "so2": 3.0,
    "no2":  17.0, "co":  300.0, "o3":  89.0
  },
  "weather": {
    "temp":           -0.5,
    "pressure":       1024.5,
    "dew_point":      -21.4,
    "rainfall":       0.0,
    "wind_direction": "NNW",
    "wind_speed":     5.7
  }
}
```

#### Recommended Indexes (production MongoDB)

| Index | Type | Supports |
|-------|------|----------|
| `station` + `timestamp` | Compound | Latest-record and date-range queries |
| `timestamp` | Single field | Global time-range scans |
| `pollutants.pm25` | Single field | Threshold / alert queries |

#### SQL vs MongoDB — Design Comparison

| Aspect | MySQL (`readings`) | MongoDB (`air_quality_readings`) |
|--------|-------------------|----------------------------------|
| Data model | Flat row, 14 columns | Nested sub-documents |
| Station reference | FK to `stations` table | Embedded string field |
| Schema enforcement | DDL constraints | Application-level / JSON Schema validation |
| Query style | JOIN-based SQL | `find()` + aggregation pipeline |
| Best suited for | Structured analytics, multi-table JOINs | Flexible retrieval, document-oriented access |

In [ ]:
# ── MongoDB connection ────────────────────────────────────────────────────────
#
# Option A (default) — mongomock: no server required, runs in-process.
# Option B (faster)  — real MongoDB on Kaggle: uncomment the two lines below
#                      and change MongoClient() to MongoClient('localhost', 27017)
#
# !sudo apt-get install -y mongodb -qq
# !sudo service mongodb start

client     = mongomock.MongoClient()
db_mongo   = client['beijing_air_quality']
collection = db_mongo['air_quality_readings']

print('MongoDB connection established.')
print(f'  Engine    : mongomock (pymongo-compatible)')
print(f'  Database  : {db_mongo.name}')
print(f'  Collection: {collection.name}')

In [ ]:
# ── Convert DataFrame → list of documents ────────────────────────────────────
# Using to_dict('records') is ~40x faster than iterrows() for 420 K rows.

def _safe(v):
    """Return None instead of NaN so documents serialise correctly."""
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

print(f'Converting {len(df_raw):,} rows to MongoDB documents …')
t0 = time.time()

raw_records = df_raw.to_dict('records')

documents = [
    {
        'station':   r['station'],
        'timestamp': r['timestamp'].to_pydatetime(),
        'year':  int(r['year']),
        'month': int(r['month']),
        'day':   int(r['day']),
        'hour':  int(r['hour']),
        'pollutants': {
            'pm25': _safe(r['PM2.5']),
            'pm10': _safe(r['PM10']),
            'so2':  _safe(r['SO2']),
            'no2':  _safe(r['NO2']),
            'co':   _safe(r['CO']),
            'o3':   _safe(r['O3']),
        },
        'weather': {
            'temp':           _safe(r['TEMP']),
            'pressure':       _safe(r['PRES']),
            'dew_point':      _safe(r['DEWP']),
            'rainfall':       _safe(r['RAIN']),
            'wind_direction': r['wd'] if isinstance(r['wd'], str) else None,
            'wind_speed':     _safe(r['WSPM']),
        },
    }
    for r in raw_records
]

t1 = time.time()
print(f'Conversion complete in {t1 - t0:.1f}s')

# ── Insert in batches of 10 000 ───────────────────────────────────────────────
BATCH = 10_000
inserted = 0
print(f'Inserting {len(documents):,} documents …')

for start in range(0, len(documents), BATCH):
    collection.insert_many(documents[start : start + BATCH], ordered=False)
    inserted += min(BATCH, len(documents) - start)
    print(f'  {inserted:>7,} / {len(documents):,}', end='\r')

t2 = time.time()
print(f'\nInsertion complete in {t2 - t1:.1f}s')
print(f'Total documents in collection: {collection.count_documents({}):,}')

In [ ]:
# Show three sample documents from the Dongsi station
print('Sample documents from air_quality_readings (Dongsi station):\n')
samples = list(collection.find({'station': 'Dongsi'}).limit(3))
for doc in samples:
    doc.pop('_id', None)          # remove ObjectId for clean display
    print(json.dumps(doc, indent=2, default=str))
    print('─' * 45)

---
## MongoDB Queries

| # | Query type | Description |
|---|-----------|-------------|
| MQ-1 | `find().sort().limit()` | Latest reading for a specific station |
| MQ-2 | `find()` with `$gte`/`$lte` | All documents within a date range |
| MQ-3 | Aggregation pipeline (`$group`) | Average PM2.5 per station |
| MQ-4 | Aggregation pipeline (`$group` + `$cond`) | Hazardous-reading count and percentage per station |

In [ ]:
# ── MQ-1: Latest reading for Dongsi station ───────────────────────────────────
print('MongoDB Query 1: Latest reading for Dongsi station')
print('-' * 60)
print("""
db.air_quality_readings.find(
    { "station": "Dongsi" },
    { "_id": 0, "station": 1, "timestamp": 1,
      "pollutants": 1, "weather.temp": 1, "weather.wind_direction": 1 }
).sort({ "timestamp": -1 }).limit(1)
""")

result_mq1 = list(
    collection.find(
        {'station': 'Dongsi'},
        {'_id': 0, 'station': 1, 'timestamp': 1,
         'pollutants': 1, 'weather': 1}
    ).sort('timestamp', -1).limit(1)
)

print('Result:')
for doc in result_mq1:
    print(json.dumps(doc, indent=2, default=str))

In [ ]:
# ── MQ-2: Date-range query — Dongsi, 1–7 January 2016 ────────────────────────
start_dt = datetime(2016, 1, 1,  0, 0, 0)
end_dt   = datetime(2016, 1, 7, 23, 0, 0)

print('MongoDB Query 2: Date-range — Dongsi, 1–7 January 2016')
print('-' * 60)
print(f"""
db.air_quality_readings.find(
    {{
        "station": "Dongsi",
        "timestamp": {{
            "$gte": ISODate("{start_dt.isoformat()}"),
            "$lte": ISODate("{end_dt.isoformat()}")
        }}
    }},
    {{ "_id": 0, "station": 1, "timestamp": 1,
      "pollutants.pm25": 1, "pollutants.pm10": 1,
      "weather.temp": 1, "weather.wind_direction": 1 }}
).sort({{ "timestamp": 1 }})
""")

cur_mq2 = collection.find(
    {
        'station':   'Dongsi',
        'timestamp': {'$gte': start_dt, '$lte': end_dt}
    },
    {'_id': 0, 'station': 1, 'timestamp': 1,
     'pollutants': 1, 'weather': 1}
).sort('timestamp', 1)

result_mq2 = list(cur_mq2)
print(f'Total documents returned: {len(result_mq2)}')

rows = [
    {
        'station':        d['station'],
        'timestamp':      d['timestamp'],
        'pm25':           d['pollutants'].get('pm25'),
        'pm10':           d['pollutants'].get('pm10'),
        'temp':           d['weather'].get('temp'),
        'wind_direction': d['weather'].get('wind_direction'),
    }
    for d in result_mq2
]
mq2_df = pd.DataFrame(rows)
print('\nFirst 10 rows:')
display(mq2_df.head(10))

In [ ]:
# ── MQ-3: Average PM2.5 per station (aggregation pipeline) ───────────────────
print('MongoDB Query 3: Average PM2.5 per station')
print('-' * 60)
print("""
db.air_quality_readings.aggregate([
    { "$match": { "pollutants.pm25": { "$ne": null } } },
    { "$group": {
        "_id":       "$station",
        "avg_pm25":  { "$avg": "$pollutants.pm25" },
        "max_pm25":  { "$max": "$pollutants.pm25" },
        "doc_count": { "$sum": 1 }
    }},
    { "$sort": { "avg_pm25": -1 } }
])
""")

pipeline_mq3 = [
    {'$match': {'pollutants.pm25': {'$ne': None}}},
    {'$group': {
        '_id':       '$station',
        'avg_pm25':  {'$avg': '$pollutants.pm25'},
        'max_pm25':  {'$max': '$pollutants.pm25'},
        'doc_count': {'$sum': 1},
    }},
    {'$sort': {'avg_pm25': -1}},
]

result_mq3 = list(collection.aggregate(pipeline_mq3))

mq3_df = pd.DataFrame([
    {
        'station':   r['_id'],
        'avg_pm25':  round(r['avg_pm25'], 2),
        'max_pm25':  round(r['max_pm25'], 2),
        'doc_count': r['doc_count'],
    }
    for r in result_mq3
])

print('Result:')
display(mq3_df)

In [ ]:
# ── MQ-4: Hazardous PM2.5 readings per station (PM2.5 > 150 μg/m³) ───────────
print('MongoDB Query 4: Hazardous PM2.5 readings (PM2.5 > 150 μg/m³) per station')
print('-' * 60)
print("""
db.air_quality_readings.aggregate([
    { "$match": { "pollutants.pm25": { "$ne": null } } },
    { "$group": {
        "_id":       "$station",
        "total":     { "$sum": 1 },
        "hazardous": {
            "$sum": { "$cond": [ { "$gt": ["$pollutants.pm25", 150] }, 1, 0 ] }
        }
    }},
    { "$addFields": {
        "pct_hazardous": {
            "$round": [
                { "$multiply": [ { "$divide": ["$hazardous", "$total"] }, 100 ] },
                1
            ]
        }
    }},
    { "$sort": { "pct_hazardous": -1 } }
])
""")

# NOTE: mongomock does not support $round — compute percentage in Python instead
pipeline_mq4 = [
    {'$match': {'pollutants.pm25': {'$ne': None}}},
    {'$group': {
        '_id':       '$station',
        'total':     {'$sum': 1},
        'hazardous': {
            '$sum': {'$cond': [{'$gt': ['$pollutants.pm25', 150]}, 1, 0]}
        },
    }},
    {'$sort': {'hazardous': -1}},
]

result_mq4 = list(collection.aggregate(pipeline_mq4))

mq4_df = pd.DataFrame([
    {
        'station':            r['_id'],
        'total_readings':     r['total'],
        'hazardous_readings': r['hazardous'],
        'pct_hazardous':      round(r['hazardous'] / r['total'] * 100, 1),
    }
    for r in result_mq4
])

mq4_df = mq4_df.sort_values('pct_hazardous', ascending=False).reset_index(drop=True)

print('Result:')
display(mq4_df)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(mq4_df['station'][::-1], mq4_df['pct_hazardous'][::-1],
               color='#e74c3c', edgecolor='#c0392b')
ax.set_xlabel('% of readings with PM2.5 > 150 μg/m³')
ax.set_title('Hazardous PM2.5 Readings (> 150 μg/m³) per Station')
for bar, pct in zip(bars, mq4_df['pct_hazardous'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{pct}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('mongo_q4_hazardous_pm25.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

### Databases Implemented

| Database | Engine | Tables / Collections | Total records |
|----------|--------|---------------------|---------------|
| Relational | MySQL (SQLite) | `stations`, `readings`, `pollutant_reference` | 420 780 rows |
| Document | MongoDB (mongomock) | `air_quality_readings` | 420 768 documents |

### Queries Executed

| ID | Database | Type | Description |
|----|----------|------|-------------|
| Q1 | MySQL/SQLite | SELECT + subquery | Latest reading per station |
| Q2 | MySQL/SQLite | SELECT + BETWEEN | Date-range filter (Dongsi, Jan 1–7 2016) |
| Q3 | MySQL/SQLite | GROUP BY + aggregate | Average/max PM2.5 ranked by station |
| Q4 | MySQL/SQLite | GROUP BY + date function | Monthly PM2.5 and temperature trend |
| Q5 | MySQL/SQLite | CROSS JOIN + CASE | WHO 2021 guideline exceedance analysis |
| MQ-1 | MongoDB | `find().sort().limit()` | Latest document for a station |
| MQ-2 | MongoDB | `find()` + `$gte`/`$lte` | Date-range document retrieval |
| MQ-3 | MongoDB | Aggregation `$group` | Average PM2.5 per station |
| MQ-4 | MongoDB | Aggregation `$group` + `$cond` | Hazardous-reading % per station |

### Key Findings from Queries

- **Q3/MQ-3**: Dongsi and Wanshouxigong consistently rank as the most polluted stations; Dingling and Huairou (suburban/rural) record the lowest average PM2.5.
- **Q4**: Winter months (December–February) show markedly higher PM2.5 than summer, consistent with coal heating and lower wind speeds during temperature inversions.
- **Q5**: Every station exceeds the WHO 24-hour PM2.5 guideline (15 μg/m³) in more than 60% of hourly observations, and all station means exceed the WHO annual guideline (5 μg/m³) by at least tenfold.
- **MQ-4**: Wanshouxigong has the highest share of truly hazardous hours (PM2.5 > 150 μg/m³); Dingling has the lowest.